# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2, sum as sum_spark, lower as lower_spark, upper as upper_spark, countDistinct, first, dense_rank, isnan, lead, greatest
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
CATALOG = "workspace"
SILVER_SCHEMA = "silver"

SILVER_LAPS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.laps"
)

SILVER_PIT_STOPS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.pit_stops"
)

In [0]:
def prepare_pit_stop_candidates():
    laps_df = spark.table(
        SILVER_LAPS_TABLE
    )

    lap_window = (
        Window
        .partitionBy(
            "season",
            "round",
            "driver_id"
        )
        .orderBy(
            "lap_number"
        )
    )

    candidates_df = (
        laps_df

        # Next lap identity
        .withColumn(
            "_next_lap_number",
            lead("lap_number").over(lap_window)
        )

        # Next lap pit exit
        .withColumn(
            "_next_pit_out_time_ms",
            lead("pit_out_time_ms").over(lap_window)
        )

        # Next stint
        .withColumn(
            "_next_stint_number",
            lead("stint_number").over(lap_window)
        )

        # Next tyre
        .withColumn(
            "_next_compound",
            lead("compound").over(lap_window)
        )

        .withColumn(
            "_next_tyre_life_laps",
            lead("tyre_life_laps").over(lap_window)
        )

        # Quality
        .withColumn(
            "_next_is_accurate",
            lead("is_accurate").over(lap_window)
        )

        # Lineage
        .withColumn(
            "_next_source_modified_at",
            lead("source_modified_at").over(lap_window)
        )

        .withColumn(
            "_next_bronze_ingested_at",
            lead("bronze_ingested_at").over(lap_window)
        )

        .withColumn(
            "_next_silver_updated_at",
            lead("silver_updated_at").over(lap_window)
        )
    )

    return candidates_df

In [0]:
def transform_pit_stops(candidates_df):

    pit_entries = (
        candidates_df
        .filter(
            col("pit_in_time_ms").isNotNull()
        )
    )

    is_complete = (
        (
            col("_next_lap_number")
            ==
            col("lap_number") + 1
        )
        &
        col("_next_pit_out_time_ms").isNotNull()
    )

    pit_stops_df = (
        pit_entries
        .select(
            "season",
            "round",
            "driver_id",
            "driver_number",
            "constructor_id",

            col("lap_number")
                .alias("pit_in_lap"),

            when(
                is_complete,
                col("_next_lap_number")
            )
            .otherwise(
                lit(None).cast("int")
            )
            .alias("pit_out_lap"),

            col("pit_in_time_ms"),

            when(
                is_complete,
                col("_next_pit_out_time_ms")
            )
            .otherwise(
                lit(None).cast("long")
            )
            .alias("pit_out_time_ms"),

            when(
                is_complete,
                (
                    col("_next_pit_out_time_ms")
                    -
                    col("pit_in_time_ms")
                )
            )
            .otherwise(
                lit(None).cast("long")
            )
            .alias("pit_lane_duration_ms"),

            col("stint_number")
                .alias("stint_before"),

            when(
                is_complete,
                col("_next_stint_number")
            )
            .otherwise(
                lit(None).cast("int")
            )
            .alias("stint_after"),

            col("compound")
                .alias("compound_before"),

            when(
                is_complete,
                col("_next_compound")
            )
            .otherwise(
                lit(None).cast("string")
            )
            .alias("compound_after"),

            col("tyre_life_laps")
                .alias("tyre_life_before_laps"),

            when(
                is_complete,
                col("_next_tyre_life_laps")
            )
            .otherwise(
                lit(None).cast("int")
            )
            .alias("tyre_life_after_laps"),

            is_complete.alias("is_complete"),

            when(
                is_complete,
                col("is_accurate")
                &
                col("_next_is_accurate")
            )
            .otherwise(
                col("is_accurate")
            )
            .alias("is_accurate"),

            "source_file",

            greatest(
                col("source_modified_at"),
                col("_next_source_modified_at")
            ).alias("source_modified_at"),

            greatest(
                col("bronze_ingested_at"),
                col("_next_bronze_ingested_at")
            ).alias("bronze_ingested_at"),

            greatest(
                col("silver_updated_at"),
                col("_next_silver_updated_at")
            ).alias("source_silver_updated_at")
        )
    )

    stop_window = (
        Window
        .partitionBy(
            "season",
            "round",
            "driver_id"
        )
        .orderBy(
            "pit_in_lap"
        )
    )

    return (
        pit_stops_df
        .withColumn(
            "stop_number",
            row_number().over(stop_window)
        )
        .select(
            "season",
            "round",
            "driver_id",
            "driver_number",
            "constructor_id",
            "stop_number",
            "pit_in_lap",
            "pit_out_lap",
            "pit_in_time_ms",
            "pit_out_time_ms",
            "pit_lane_duration_ms",
            "stint_before",
            "stint_after",
            "compound_before",
            "compound_after",
            "tyre_life_before_laps",
            "tyre_life_after_laps",
            "is_complete",
            "is_accurate",
            "source_file",
            "source_modified_at",
            "bronze_ingested_at",
            "source_silver_updated_at"
        )
    )

In [0]:
def validate_pit_stops(df):
    errors = {}

    required_validation = (
        df
        .agg(
            sum_spark(
                when(
                    col("season").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_season"),

            sum_spark(
                when(
                    col("round").isNull()
                    |
                    (col("round") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_round"),

            sum_spark(
                when(
                    col("driver_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_driver_id"),

            sum_spark(
                when(
                    col("constructor_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_constructor_id"),

            sum_spark(
                when(
                    col("pit_in_lap").isNull()
                    |
                    (col("pit_in_lap") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_pit_in_lap"),

            sum_spark(
                when(
                    col("pit_in_time_ms").isNull()
                    |
                    (col("pit_in_time_ms") < 0),
                    1
                ).otherwise(0)
            ).alias("invalid_pit_in_time"),

            sum_spark(
                when(
                    col("stop_number") <= 0,
                    1
                ).otherwise(0)
            ).alias("invalid_stop_number")
        )
        .first()
        .asDict()
    )

    errors.update({
        key: value or 0
        for key, value in required_validation.items()
        if (value or 0) > 0
    })

    # ------------------------------------------------------
    # Natural key
    # ------------------------------------------------------

    duplicate_entries = (
        df
        .groupBy(
            "season",
            "round",
            "driver_id",
            "pit_in_lap"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_entries > 0:
        errors["duplicate_pit_entries"] = (
            duplicate_entries
        )

    # ------------------------------------------------------
    # stop_number must also be unique
    # ------------------------------------------------------

    duplicate_stop_numbers = (
        df
        .groupBy(
            "season",
            "round",
            "driver_id",
            "stop_number"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_stop_numbers > 0:
        errors["duplicate_stop_number"] = (
            duplicate_stop_numbers
        )

    # ------------------------------------------------------
    # Complete stop consistency
    # ------------------------------------------------------

    invalid_complete_stops = (
        df
        .filter(
            col("is_complete")
            &
            (
                col("pit_out_lap").isNull()
                |
                col("pit_out_time_ms").isNull()
                |
                col("pit_lane_duration_ms").isNull()
                |
                (
                    col("pit_out_lap")
                    !=
                    col("pit_in_lap") + 1
                )
                |
                (
                    col("pit_out_time_ms")
                    <=
                    col("pit_in_time_ms")
                )
                |
                (
                    col("pit_lane_duration_ms")
                    <= 0
                )
            )
        )
        .count()
    )

    if invalid_complete_stops > 0:
        errors["invalid_complete_stop"] = (
            invalid_complete_stops
        )

    if errors:
        raise ValueError(
            f"Silver pit_stops validation failed: {errors}"
        )

    print(
        f"Validation OK: {df.count()} pit entries ready for Silver."
    )

In [0]:
def show_incomplete_pit_stops(df):
    incomplete_df = (
        df
        .filter(
            ~col("is_complete")
        )
    )

    incomplete_count = (
        incomplete_df.count()
    )

    if incomplete_count > 0:
        print(
            f"WARNING: {incomplete_count} pit entries "
            "do not have a matching pit exit on the next lap."
        )

        display(
            incomplete_df.select(
                "season",
                "round",
                "driver_id",
                "pit_in_lap",
                "pit_in_time_ms",
                "stint_before",
                "compound_before"
            )
        )

In [0]:
def add_pit_stops_hash(df):
    business_columns = [
        "season",
        "round",
        "driver_id",
        "driver_number",
        "constructor_id",
        "stop_number",
        "pit_in_lap",
        "pit_out_lap",
        "pit_in_time_ms",
        "pit_out_time_ms",
        "pit_lane_duration_ms",
        "stint_before",
        "stint_after",
        "compound_before",
        "compound_after",
        "tyre_life_before_laps",
        "tyre_life_after_laps",
        "is_complete",
        "is_accurate"
    ]

    hash_expression = concat_ws(
        "||",
        *[
            coalesce(
                col(column).cast("string"),
                lit("<NULL>")
            )
            for column in business_columns
        ]
    )

    return (
        df
        .withColumn(
            "record_hash",
            sha2(
                hash_expression,
                256
            )
        )
        .withColumn(
            "silver_updated_at",
            current_timestamp()
        )
    )

In [0]:
def get_pit_stop_race_scope():
    return (
        spark.table(SILVER_LAPS_TABLE)
        .select(
            "season",
            "round"
        )
        .distinct()
    )

In [0]:
def build_race_scope_condition(scope_df):
    races = scope_df.collect()

    if not races:
        raise ValueError(
            "No races available for pit stop synchronization."
        )

    return " OR ".join(
        [
            (
                f"(target.season = {row['season']} "
                f"AND target.round = {row['round']})"
            )
            for row in races
        ]
    )

In [0]:
def merge_pit_stops(df, race_scope_df):

    if not spark.catalog.tableExists(
        SILVER_PIT_STOPS_TABLE
    ):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(
                SILVER_PIT_STOPS_TABLE
            )
        )

        print(
            f"Created {SILVER_PIT_STOPS_TABLE}"
        )
        return

    target = DeltaTable.forName(
        spark,
        SILVER_PIT_STOPS_TABLE
    )

    race_scope = build_race_scope_condition(
        race_scope_df
    )

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            """
            target.season = source.season
            AND target.round = source.round
            AND target.driver_id = source.driver_id
            AND target.pit_in_lap = source.pit_in_lap
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target.record_hash <> source.record_hash
            """
        )
        .whenNotMatchedInsertAll()

        .whenNotMatchedBySourceDelete(
            condition=race_scope
        )

        .execute()
    )

    print(
        f"Merged data into {SILVER_PIT_STOPS_TABLE}"
    )

In [0]:
pit_stop_candidates_df = (
    prepare_pit_stop_candidates()
)

pit_stops_df = (
    transform_pit_stops(
        pit_stop_candidates_df
    )
)

validate_pit_stops(
    pit_stops_df
)

show_incomplete_pit_stops(
    pit_stops_df
)

pit_stops_df = (
    add_pit_stops_hash(
        pit_stops_df
    )
)

race_scope_df = (
    get_pit_stop_race_scope()
)

merge_pit_stops(
    pit_stops_df,
    race_scope_df
)